# Compare ARRTQ, and Balanced ARRTQ

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))  # add parent directory to search path

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from core.process import Process
from core.scheduler_arrtq import arrtq
from core.scheduler_rr import round_robin
from core.scheduler_modified_ARRTQ import arrtq_balanced
from core.process_generator import generate_processes
import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import copy

In [17]:
context_switch_time = 1
sizes = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
num_trials = 50  # Run 50 trials per dataset size.... took about 40 mins to run, could start with smaller trials and improve gradually
seed_base = 5

# Results - store all trials first
all_results = {'ARRTQ': {}, 'Balanced ARRTQ': {}}

for size in sizes:
    all_results['ARRTQ'][size] = []
    all_results['Balanced ARRTQ'][size] = []

seed_counter = 0
for i, size in enumerate(sizes):
    print(f"Running {num_trials} trials for {size} processes...")
    
    for trial in range(num_trials):
        # Same data sent to both the algorithms
        processes = generate_processes(size, seed=seed_counter)
        seed_counter += 1
        
        arrtq_metrics = arrtq(copy.deepcopy(processes), context_switch_time)
        all_results['ARRTQ'][size].append(arrtq_metrics)
        
        arrtq_balanced_metrics = arrtq_balanced(copy.deepcopy(processes), context_switch_time, c=0.9)
        all_results['Balanced ARRTQ'][size].append(arrtq_balanced_metrics)
    
    print(f"  ✓ Completed {num_trials} trials")

# Calculate averages across trials
summary = []

for size in sizes:
    # Average metrics for ARRTQ
    arrtq_avg = {
        "Avg_TAT": np.mean([m.get("average_turnaround_time", np.nan) for m in all_results['ARRTQ'][size]]),
        "Avg_WT": np.mean([m.get("average_waiting_time", np.nan) for m in all_results['ARRTQ'][size]]),
        "Avg_FRT": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['ARRTQ'][size]]),
        "Throughput": np.mean([m.get("throughput", np.nan) for m in all_results['ARRTQ'][size]]),
        "CPU_Util": np.mean([m.get("cpu_utilization", np.nan) for m in all_results['ARRTQ'][size]]),
        "Context_Switches": np.mean([m.get("context_switches", np.nan) for m in all_results['ARRTQ'][size]]),

        "FRT_CI_lower": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['ARRTQ'][size]])
        - 2 * (np.std([m.get("average_first_response_time", np.nan) for m in all_results['ARRTQ'][size]], ddof=1)
               / np.sqrt(len(all_results['ARRTQ'][size]))),

        "FRT_CI_upper": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['ARRTQ'][size]])
        + 2 * (np.std([m.get("average_first_response_time", np.nan) for m in all_results['ARRTQ'][size]], ddof=1)
               / np.sqrt(len(all_results['ARRTQ'][size]))),
    }
    
    # Average metrics for Balanced ARRTQ
    balanced_avg = {
        "Avg_TAT": np.mean([m.get("average_turnaround_time", np.nan) for m in all_results['Balanced ARRTQ'][size]]),
        "Avg_WT": np.mean([m.get("average_waiting_time", np.nan) for m in all_results['Balanced ARRTQ'][size]]),
        "Avg_FRT": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['Balanced ARRTQ'][size]]),
        "Throughput": np.mean([m.get("throughput", np.nan) for m in all_results['Balanced ARRTQ'][size]]),
        "CPU_Util": np.mean([m.get("cpu_utilization", np.nan) for m in all_results['Balanced ARRTQ'][size]]),
        "Context_Switches": np.mean([m.get("context_switches", np.nan) for m in all_results['Balanced ARRTQ'][size]]),

        "FRT_CI_lower": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['Balanced ARRTQ'][size]])
        - 2 * (np.std([m.get("average_first_response_time", np.nan) for m in all_results['Balanced ARRTQ'][size]], ddof=1)
               / np.sqrt(len(all_results['Balanced ARRTQ'][size]))),
               
        "FRT_CI_upper": np.mean([m.get("average_first_response_time", np.nan) for m in all_results['Balanced ARRTQ'][size]])
        + 2 * (np.std([m.get("average_first_response_time", np.nan) for m in all_results['Balanced ARRTQ'][size]], ddof=1)
               / np.sqrt(len(all_results['Balanced ARRTQ'][size]))),
    }
    
    summary.append({"Processes": size, "Algorithm": "ARRTQ", **arrtq_avg})
    summary.append({"Processes": size, "Algorithm": "Balanced ARRTQ", **balanced_avg})

# DataFrame
summary_df = pd.DataFrame(summary)
print(f"\nAverage Results across {num_trials} trials per dataset size:")
display(summary_df)


Running 50 trials for 1000 processes...
  ✓ Completed 50 trials
Running 50 trials for 2000 processes...
  ✓ Completed 50 trials
Running 50 trials for 3000 processes...
  ✓ Completed 50 trials
Running 50 trials for 4000 processes...
  ✓ Completed 50 trials
Running 50 trials for 5000 processes...
  ✓ Completed 50 trials
Running 50 trials for 6000 processes...
  ✓ Completed 50 trials
Running 50 trials for 7000 processes...
  ✓ Completed 50 trials
Running 50 trials for 8000 processes...
  ✓ Completed 50 trials
Running 50 trials for 9000 processes...
  ✓ Completed 50 trials
Running 50 trials for 10000 processes...
  ✓ Completed 50 trials

Average Results across 50 trials per dataset size:


,Processes,Algorithm,Avg_TAT,Avg_WT,Avg_FRT,Throughput,CPU_Util,Context_Switches,FRT_CI_lower,FRT_CI_upper
0,1000,ARRTQ,5702.759621,5691.799501,4082.841176,0.076473,99.980915,2120.98,4018.969078,4146.713273
1,1000,Balanced ARRTQ,5772.316754,5761.356634,3260.492963,0.076917,99.980818,2044.96,3231.626967,3289.358959
2,2000,ARRTQ,11501.339260,11490.296500,8311.232772,0.075888,99.991338,4270.92,8188.569152,8433.896391
3,2000,Balanced ARRTQ,11651.809411,11640.766651,6550.281849,0.076356,99.991280,4109.12,6520.809186,6579.754511
4,3000,ARRTQ,17095.356452,17084.364639,12505.089108,0.076199,99.993101,6397.78,12362.119161,12648.059054
5,3000,Balanced ARRTQ,17331.149260,17320.157446,9739.085066,0.076663,99.993058,6159.40,9694.035762,9784.134370
6,4000,ARRTQ,22963.044567,22952.058102,16357.786497,0.076194,99.994937,8554.44,16128.695941,16586.877053
7,4000,Balanced ARRTQ,23164.457361,23153.470896,12996.568366,0.076683,99.994906,8219.86,12946.848426,13046.288306
8,5000,ARRTQ,28719.408583,28708.414103,20441.372517,0.076138,99.996137,10699.16,20143.099915,20739.645119
9,5000,Balanced ARRTQ,28895.803382,28884.808902,16221.337801,0.076660,99.996110,10252.60,16183.665532,16259.010071


In [2]:
summary_df.to_csv("tmp.csv", index=False)

NameError: name 'summary_df' is not defined

In [1]:
# --- CONFIGURATION ---
df = summary_df.copy()
metrics = ['Avg_FRT', 'Avg_TAT', 'Avg_WT', 'Context_Switches']
processes = sorted(df['Processes'].unique())
x = np.arange(len(processes))
width = 0.35

for metric in metrics:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Split data by algorithm
    arrtq_data = df[df['Algorithm'] == 'ARRTQ'].sort_values('Processes')
    bal_data = df[df['Algorithm'] == 'Balanced ARRTQ'].sort_values('Processes')
    
    # --- CONDITIONAL PLOTTING ---
    if metric == 'Avg_FRT':
        # Calculate asymmetric error bars ONLY for FRT
        yerr_arrtq = [
            (arrtq_data[metric] - arrtq_data['FRT_CI_lower']).values,
            (arrtq_data['FRT_CI_upper'] - arrtq_data[metric]).values
        ]
        yerr_bal = [
            (bal_data[metric] - bal_data['FRT_CI_lower']).values,
            (bal_data['FRT_CI_upper'] - bal_data[metric]).values
        ]
        
        rects1 = ax.bar(x - width/2, arrtq_data[metric], width, label='ARRTQ', yerr=yerr_arrtq, capsize=5)
        rects2 = ax.bar(x + width/2, bal_data[metric], width, label='Balanced ARRTQ', yerr=yerr_bal, capsize=5)
        
    else:
        # Standard plot for other metrics
        rects1 = ax.bar(x - width/2, arrtq_data[metric], width, label='ARRTQ')
        rects2 = ax.bar(x + width/2, bal_data[metric], width, label='Balanced ARRTQ')
    
    # --- FORMATTING ---
    ax.set_xlabel('Processes')
    ax.set_ylabel(metric)
    ax.set_title(f'Comparison of {metric} by Algorithm')
    ax.set_xticks(x)
    ax.set_xticklabels(processes)
    ax.legend()
    
    ax.bar_label(rects1, padding=3, fmt='%.2f', rotation=90)
    ax.bar_label(rects2, padding=3, fmt='%.2f', rotation=90)

    # Increase Y-limit to prevent labels from being cut off
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 1.15)

    plt.tight_layout()
    plt.savefig(f'{metric}_vertical.png')
    plt.show()

NameError: name 'summary_df' is not defined